In [ ]:
import time
from statistics import median
from urllib.parse import urlsplit
from pathlib import Path
import hashlib
import subprocess
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
import requests
from databricks.sdk import WorkspaceClient

for name, default in {
    "project_id": "lakebase-data-api-example",
    "source_catalog": "lakebase_data_api_example",
    "warehouse_id": "",
    "secret_scope": "",
    "secret_key": "",
    "oauth_client_id": "",
    "editor_ms": "",
    "cdf_catalog": "",
    "cdf_schema": "",
}.items():
    dbutils.widgets.text(name, default)

settings = dbutils.widgets.getAll()
workspace = WorkspaceClient()
project_id = settings["project_id"]
branch = f"projects/{project_id}/branches/production"
endpoint = f"{branch}/endpoints/primary"
database = f"{branch}/databases/databricks-postgres"
control_api = "/api/2.0/postgres/"
primary = workspace.api_client.do("GET", control_api + endpoint)["status"]
if primary.get("no_suspension") or float(primary["suspend_timeout_duration"].removesuffix("s")) != 60:
    raise ValueError("Set the primary endpoint's idle timeout to 60 seconds before benchmarking.")
if (primary["autoscaling_limit_min_cu"], primary["autoscaling_limit_max_cu"]) != (1, 1):
    raise ValueError("Set both compute limits to 1 CU for this benchmark.")
pg_host = primary["hosts"]["host"]
fields = ["order_id", "tenant_id", "amount_cents"]
ids = [32000 + 7919 * i for i in range(8)]
warm_where = "order_id BETWEEN 37 AND 41 ORDER BY order_id DESC LIMIT 2"
columns = ", ".join(fields)

In [ ]:
def data_api_session():
    info = workspace.api_client.do("GET", control_api + database + "/data-api")
    base = info["status"]["url"].rstrip("/")
    if urlsplit(base).scheme != "https" or urlsplit(base).hostname != pg_host:
        raise ValueError("The Data API URL does not match the selected Lakebase endpoint.")
    if settings["oauth_client_id"]:
        secret = dbutils.secrets.get(settings["secret_scope"], settings["secret_key"])
        identity = WorkspaceClient(
            host=workspace.config.host, client_id=settings["oauth_client_id"],
            client_secret=secret, auth_type="oauth-m2m",
        )
        token = identity.api_client.do("POST", control_api + "credentials", body={"endpoint": endpoint})["token"]
    elif settings["secret_scope"] and settings["secret_key"]:
        token = dbutils.secrets.get(settings["secret_scope"], settings["secret_key"])
    else:
        token = workspace.api_client.do("POST", control_api + "credentials", body={"endpoint": endpoint})["token"]
    headers = {"Authorization": f"Bearer {token}"}
    session = requests.Session()
    session.headers.update(headers)
    return session, base + "/public/orders"

def read_orders(filters):
    response = http.get(orders_url, params=[("select", ",".join(fields)), *filters], timeout=30)
    response.raise_for_status()
    return response.json()

http = orders_url = None
try:
    http, orders_url = data_api_session()
    order = read_orders([("order_id", "eq.42")])
    if len(order) != 1 or int(order[0]["order_id"]) != 42:
        raise ValueError("The Data API role cannot see the seeded order 42.")
    display(pd.DataFrame(order, columns=fields))
except requests.RequestException as error:
    http = orders_url = None
    status_code = getattr(error.response, "status_code", None)
    print("Data API request failed:", type(error).__name__, status_code)
    print("Use a non-owner identity whose Postgres role is granted to authenticator and can read public.orders.")

In [ ]:
def expected(key):
    return (key, key % 32, key * 17 % 100000)

samples = {}

def measure(label, read):
    values = []
    for key in ids:
        if read(None) != [expected(41), expected(40)]:
            raise ValueError(f"{label}: warm-up result differs from the fixture")
        started = time.perf_counter_ns()
        rows = read(key)
        elapsed_ms = (time.perf_counter_ns() - started) / 1e6
        if rows != [expected(key)]:
            raise ValueError(f"{label}: incorrect row for order {key}")
        values.append(elapsed_ms)
    samples[label] = values
    print(f"{label}: {median(values):.1f} ms median for {len(values)} verified reads")

def jdbc_timings():
    version = "42.7.9"
    checksum = "88f1fc3992e80ec3b048f798030e9a014aa4783c40afb56d3e7a87ee0adf166f"
    url = f"https://repo.maven.apache.org/maven2/org/postgresql/postgresql/{version}/postgresql-{version}.jar"
    driver = requests.get(url, timeout=30)
    driver.raise_for_status()
    if hashlib.sha256(driver.content).hexdigest() != checksum:
        raise ValueError("The pgJDBC driver failed its SHA-256 check")
    source = Path.cwd() / "Lookup.java"
    if not source.is_file():
        raise FileNotFoundError("Run from the synced Git folder containing Lookup.java")
    user = workspace.current_user.me().user_name
    password = workspace.api_client.do("POST", control_api + "credentials", body={"endpoint": endpoint})["token"]
    with tempfile.TemporaryDirectory() as work:
        jar = Path(work) / "driver.jar"
        jar.write_bytes(driver.content)
        run = subprocess.run(
            ["java", "-cp", str(jar), str(source)],
            input="\n".join([pg_host, user, password, ""]),
            text=True, capture_output=True, timeout=180,
        )
    if run.returncode:
        reason = next((line for line in run.stderr.splitlines() if "Exception" in line or "error:" in line), "JVM unavailable")
        raise RuntimeError(reason.replace(password, "[redacted]")[:220])
    lines = run.stdout.strip().splitlines()
    if lines[:1] != ["PK=order_id"] or len(lines) != 9:
        raise ValueError("JDBC returned incomplete results")
    keys, values = zip(*(line.split(",") for line in lines[1:]))
    if list(map(int, keys)) != ids:
        raise ValueError("JDBC results do not match the benchmark keys")
    print("Verified primary key on public.orders: order_id")
    return list(map(float, values))

try:
    samples["PostgreSQL JDBC"] = jdbc_timings()
    print(f"PostgreSQL JDBC: {median(samples['PostgreSQL JDBC']):.1f} ms median")
except (OSError, RuntimeError, subprocess.SubprocessError, requests.RequestException) as error:
    print("JDBC unavailable:", str(error)[:250])

In [ ]:
def rest_lookup(key):
    filters = ([("order_id", "gte.37"), ("order_id", "lte.41"),
                ("order", "order_id.desc"), ("limit", "2")]
               if key is None else [("order_id", f"eq.{key}")])
    return [tuple(int(row[f]) for f in fields) for row in read_orders(filters)]

if http is not None:
    measure("Lakebase Data API", rest_lookup)
    http.close()

In [ ]:
def identifier(value):
    if not value:
        raise ValueError("Enter a nonempty catalog or schema name.")
    return "`" + value.replace("`", "``") + "`"

def execute_sql(sql):
    if not settings["warehouse_id"]:
        raise ValueError("Set warehouse_id to the serverless SQL warehouse.")
    route = "/api/2.0/sql/statements"
    response = workspace.api_client.do("POST", route, body={
        "warehouse_id": settings["warehouse_id"], "statement": sql, "wait_timeout": "10s",
        "on_wait_timeout": "CONTINUE", "disposition": "INLINE", "format": "JSON_ARRAY",
    })
    statement_id = response["statement_id"]
    deadline = time.monotonic() + 180
    while response["status"]["state"] in {"PENDING", "RUNNING"}:
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Warehouse statement {statement_id} timed out")
        time.sleep(0.1)
        response = workspace.api_client.do("GET", f"{route}/{statement_id}")
    status = response["status"]
    if status["state"] != "SUCCEEDED":
        raise RuntimeError(f"{statement_id}: {status.get('error', {}).get('message', status['state'])}")
    manifest = response["manifest"]
    chunk = response["result"]
    rows = list(chunk.get("data_array", []))
    while chunk.get("next_chunk_index") is not None:
        index = int(chunk["next_chunk_index"])
        chunk = workspace.api_client.do("GET", f"{route}/{statement_id}/result/chunks/{index}")
        rows.extend(chunk.get("data_array", []))
    if manifest.get("truncated") or len(rows) != manifest["total_row_count"]:
        raise ValueError("Incomplete SQL response")
    return rows

def warehouse_lookup(key):
    predicate = warm_where if key is None else f"order_id = {int(key)}"
    table = identifier(settings["source_catalog"]) + ".public.orders"
    rows = execute_sql(f"SELECT {columns} FROM {table} WHERE {predicate}")
    return [tuple(map(int, row)) for row in rows]

if settings["warehouse_id"]:
    warehouse = workspace.api_client.do("GET", "/api/2.0/sql/warehouses/" + settings["warehouse_id"])
    if not warehouse.get("enable_serverless_compute"):
        raise ValueError("Select a serverless warehouse to query the registered Lakebase database.")
    reg = workspace.api_client.do("GET", control_api + "catalogs/" + settings["source_catalog"])["status"]
    if reg["branch"] != branch or reg["postgres_database"] != "databricks_postgres":
        raise ValueError("The registered catalog points to a different source database.")
    measure("Statement Execution API", warehouse_lookup)
else:
    print("Set warehouse_id to run the SQL warehouse benchmark.")

In [ ]:
print("In the Lakebase SQL Editor, run the warm-up before each lookup. Enter the eight displayed durations in editor_ms.")
for key in ids:
    print(f"SELECT {columns} FROM public.orders WHERE {warm_where};")
    print(f"SELECT {columns} FROM public.orders WHERE order_id = {key};", "Expected:", expected(key))

In [ ]:
import math

editor_text = dbutils.widgets.get("editor_ms").strip()
samples.pop("Lakebase SQL Editor", None)
if editor_text:
    editor = [float(value.strip()) for value in editor_text.split(",")]
    if len(editor) != len(ids) or any(not math.isfinite(n) or n <= 0 for n in editor):
        raise ValueError("Enter eight positive, comma-separated SQL Editor durations in milliseconds.")
    samples["Lakebase SQL Editor"] = editor

methods = ["Lakebase Data API", "PostgreSQL JDBC", "Statement Execution API", "Lakebase SQL Editor"]
summary = pd.DataFrame([
    {"Access path": name, "Reads": len(samples[name]), "Median (ms)": round(median(samples[name]), 1)}
    for name in methods if name in samples
])
display(summary)
if all(name in samples for name in methods):
    values = [median(samples[name]) for name in methods]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    bars = ax.barh(methods, values, color=["#e00000", "#f38d92", "#f38d92", "#f38d92"])
    ax.bar_label(bars, labels=[f"{v:,.1f} ms" for v in values], padding=5, fontweight="bold")
    ax.set_xlim(0, max(values) * 1.3)
    ax.invert_yaxis()
    ax.set_axisbelow(True)
    ax.grid(axis="x", linestyle="--", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("Median time to read one indexed order")
    fig.tight_layout()
    plt.show()
else:
    print("The chart requires all four methods. Re-run the missing methods and enter eight SQL Editor durations.")

In [ ]:
cdf_catalog = dbutils.widgets.get("cdf_catalog")
cdf_schema = dbutils.widgets.get("cdf_schema")
if cdf_catalog and cdf_schema:
    history = f"{identifier(cdf_catalog)}.{identifier(cdf_schema)}.lb_orders_history"
    sql = f"""SELECT order_id, amount_cents, _pg_change_type
              FROM {history}
              WHERE order_id = 42
              ORDER BY _sort_by LIMIT 20"""
    events = execute_sql(sql)
    display(pd.DataFrame(events, columns=["order_id", "amount_cents", "_pg_change_type"]))
else:
    print("Set cdf_catalog and cdf_schema to the Unity Catalog destination of your change feed.")